In [ ]:
!pip install scikeras
!pip install openpyxl
!pip install imblearn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from google.colab import files
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize
from sklearn.ensemble import AdaBoostClassifier
from scikeras.wrappers import KerasClassifier
from imblearn.over_sampling import SMOTE
import tensorflow as tf # Import tensorflow
from tensorflow.keras.layers import Input
from sklearn.model_selection import RandomizedSearchCV


uploaded = files.upload()  # Upload your file in Colab

file_name = list(uploaded.keys())[0]  # Get the name of the uploaded file
data = pd.read_excel(file_name)


X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values


scaler = StandardScaler()
X = scaler.fit_transform(X)


sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
for train_index, test_index in sss.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]


smote = SMOTE(k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)


def build_model():
        model = Sequential()
        model.add(Input(shape=(X_resampled.shape[1],)))
        model.add(Dense(32, activation='relu'))
        model.add(Dense(6, activation='softmax'))  # Output layer with 6 classes (softmax for probability)

        model.compile(optimizer='Adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        return model



keras_clf = KerasClassifier(build_fn=build_model, epochs=200, batch_size=16, verbose=0)


ada_boost = AdaBoostClassifier(estimator=keras_clf, n_estimators=50, learning_rate=0.05, algorithm="SAMME")


ada_boost.fit(X_resampled, y_resampled)
y_pred=ada_boost.predict(X_test)

y_test_binarized = label_binarize(y_test, classes=range(len(set(y))))
y_pred_binarized = label_binarize(y_pred, classes=range(len(set(y))))
# Precision, Recall, F1-Score, and AUC Score
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
auc = roc_auc_score(y_test_binarized, ada_boost.predict_proba(X_test), average='macro', multi_class='ovr')
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall: {recall:.4f}")
print(f"Macro F1 Score: {f1:.4f}")
print(f"Macro AUC Score: {auc:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))


#score = ada_boost.score(X_test, y_test)
#print(f"AdaBoost Test Accuracy: {score}")

Saving 594_peer_play.xlsx to 594_peer_play.xlsx


/usr/local/lib/python3.10/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.10/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.10/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.10/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.10/dist-packages/scikeras

Accuracy: 0.4722
Macro Precision: 0.2569
Macro Recall: 0.3589
Macro F1 Score: 0.2778
Macro AUC Score: 0.6542

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.74      0.80        19
           1       0.17      0.17      0.17         6
           2       0.00      0.00      0.00         3
           3       0.17      0.25      0.20         4
           4       0.00      0.00      0.00         3
           5       0.33      1.00      0.50         1

    accuracy                           0.47        36
   macro avg       0.26      0.36      0.28        36
weighted avg       0.52      0.47      0.49        36



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
